# 13_TRANSFORMER-LAYER-PROBES

**Role.** Analysis of frozen models (WP9), not a technique in the comparison. For every layer of the fine-tuned transformers of the comparison, and of the pretrained checkpoints they started from, it fits linear probes on mean-pooled hidden states for three targets: the extremist label, "an identity term is present", and a word-type control task. It then removes the identity-probe direction from the final layer and measures how p(extremist) moves on test posts that mention a community. The design is fixed in `docs/preregistrations/WP9_layer_probes.md`, committed before this notebook was written.

**Research questions.** RQ3: at which layer the label becomes linearly decodable, and how much of it fine-tuning adds over pretraining. RQ4: whether the linear identity direction carries label-relevant signal. It trains no post classifier, selects no threshold, model or configuration, and enters no RQ1 comparison.

**How to run on Kaggle.**
1. **Accelerator:** GPU, one T4 (16 GB) or one P100. Internet **on** (the pretrained base checkpoints are downloaded from the Hugging Face Hub).
2. **Inputs to attach** (Add data):
   - this repository as a dataset (it provides `tools/notebook_kit.py`, `tools/probe_kit.py`, `data/lexicons/identity_terms.txt`, the committed `results_summary/*/best_config.json` and `research_loop/probs/*.csv`, and the split mirror);
   - the output of `00_create_dataset_and_splits` (the `research_foundation/` folder);
   - a dataset made from `external/07_TWITTER-ROBERTA_FINE-TUNE/` with `weights/transformer_model/` and `tokenizer/`;
   - a dataset made from `external/11_MULTI-CHECKPOINT_LOGIT-POOL/` with `weights/member-<C>-seed<S>/` for C in A_anchor, B_dynabench, C_hatebert, E_twitter_large and S in 17, 30, 73 (12 folders; D and F are not needed), `tokenizer/<C>/` for the same four, and `training_logs/run_summary.csv` (the inference precision of each run).
3. **Expected runtime:** about 2.5 to 4 hours. Hidden-state extraction is about 30 minutes for the 22 model passes over 2,999 posts; the probe fits (about 280 layers x 3 tasks, 5-fold cross-validation over 4 values of C each, on the kernel's CPUs) take most of the rest; the bootstrap, the contrasts and the ablation take about 15 minutes. This is within Kaggle's 12-hour session limit.
4. **Memory:** about 3 GB of host RAM for the train and validation hidden states (float32), freed after the probe fit; under 6 GB of GPU memory (one model at a time, float16 autocast, batch 64). Nothing large is written unless `save_hidden_states` is switched on (about 2 to 3 GB, see section 11).
5. Run all cells. Section 11 writes `13_TRANSFORMER-LAYER-PROBES_repo_files.zip`.

**Outputs.**
- `results_summary/rq/rq3_layer_probes.csv`, `rq3_layer_probe_contrasts.csv`, `rq4_identity_direction_ablation.csv` and `rq3_layer_probes_run.json`: text-free, copied into the repository (schema in `docs/RESULTS_SCHEMA.md`), then checked with `python3 tools/layer_probes_report.py --check`.
- `external/13_TRANSFORMER-LAYER-PROBES/`: nothing by default. With `save_hidden_states`, per-post hidden states of the fine-tuned runs, uploaded to a Kaggle Dataset and recorded with `tools/run_manifest.py add-asset --contains-text` (section 11 prints the commands).

**Protocol.** Probes are fitted on the train split; their regularisation is chosen by cross-validation inside the train split. Validation is scored as a held-out check and used for nothing. The test split is opened once, in section 8, by `pk.begin_test_evaluation`; the probe kit refuses test rows before that. Each restored model must reproduce its committed probability artifact (tolerance 0.01) on validation before any probe is fitted, and on test before any test number is computed.

**What this notebook never shows.** Cell outputs are committed, so no cell prints, displays or plots a post, a row id or a text hash. Outputs are counts, configurations, verification results and per-layer metrics.

**Responsible use.** A research artifact for studying how classifiers of extremist content represent posts. Labels are post-level and features are text only; nothing here supports decisions about people, accounts or communities.


## 1. Bootstrap

Locate and load the shared notebook kit (`tools/notebook_kit.py`) and, from the same folder, the probe kit (`tools/probe_kit.py`), then import the libraries this notebook uses.


In [ ]:
import importlib.util
import sys
from pathlib import Path

_kit_candidates = (
    sorted(Path("/kaggle/input").rglob("notebook_kit.py"))
    if Path("/kaggle/input").exists()
    else []
)
_kit_candidates += [p / "tools" / "notebook_kit.py" for p in (Path.cwd(), *Path.cwd().parents)]
_kit_path = next((p for p in _kit_candidates if p.is_file()), None)
if _kit_path is None:
    raise FileNotFoundError(
        "notebook_kit.py not found. Attach the repository (or its tools/ folder) "
        "to this kernel as an input dataset."
    )
_kit_spec = importlib.util.spec_from_file_location("notebook_kit", _kit_path)
nk = importlib.util.module_from_spec(_kit_spec)
sys.modules["notebook_kit"] = nk
_kit_spec.loader.exec_module(nk)

_probe_kit_path = _kit_path.parent / "probe_kit.py"
if not _probe_kit_path.is_file():
    raise FileNotFoundError(
        "probe_kit.py not found next to notebook_kit.py. Attach the repository's tools/ "
        "folder to this kernel as an input dataset."
    )
_probe_kit_spec = importlib.util.spec_from_file_location("probe_kit", _probe_kit_path)
pk = importlib.util.module_from_spec(_probe_kit_spec)
sys.modules["probe_kit"] = pk
_probe_kit_spec.loader.exec_module(pk)

import gc
import html
import json
import os
import re
import time
import unicodedata
from contextlib import nullcontext

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegressionCV
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from transformers import AutoModel, AutoModelForSequenceClassification, AutoTokenizer

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
pd.set_option("display.max_columns", 50)
print("probe_kit", pk.PROBE_KIT_VERSION)


## 2. Configuration

Everything that defines the run is in `CONFIG`, as preregistered. `pk.validate_probe_config` refuses a configuration without its probe, control, bootstrap and ablation blocks or with a pooling other than the mean; `nk.bootstrap` refuses a technique name that is not the filename stem, another split or dataset version, or text previews switched on, seeds every generator and creates the output folders.

Each fine-tuned entry names the committed technique whose weights it uses; section 5 checks its checkpoint, normalisation and the pool's components against that technique's committed `best_config.json`. Each base entry names the fine-tuned run whose inputs it is served with (`paired_with`).


In [ ]:
CONFIG = {
    # Identity
    "project_name": "extremism_text_classification_comparison",
    "technique_name": "13_TRANSFORMER-LAYER-PROBES",
    "role": "probe",
    "dataset_version": "extremism_dataset_clean_v1",
    "split_version": "split_v1_stratified_70_15_15_seed30",
    "random_seed": 30,
    "sources": {
        "single": "07_TWITTER-ROBERTA_FINE-TUNE",
        "pool": "11_MULTI-CHECKPOINT_LOGIT-POOL",
    },
    # Model runs, as preregistered: 07, the 12 runs of 11's admitted primary
    # pool, and the pretrained checkpoints they started from.
    "models": [
        {"model_id": "ft_07", "kind": "fine_tuned", "source": "single", "component": "single", "seed": 30,
         "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest", "text_normalization": "identity",
         "base_id": "base_twitter_roberta_base_identity"},
        {"model_id": "ft_11_a_anchor_s17", "kind": "fine_tuned", "source": "pool", "component": "A_anchor", "seed": 17,
         "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest", "text_normalization": "minimal_twitter",
         "base_id": "base_twitter_roberta_base_minimal"},
        {"model_id": "ft_11_a_anchor_s30", "kind": "fine_tuned", "source": "pool", "component": "A_anchor", "seed": 30,
         "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest", "text_normalization": "minimal_twitter",
         "base_id": "base_twitter_roberta_base_minimal"},
        {"model_id": "ft_11_a_anchor_s73", "kind": "fine_tuned", "source": "pool", "component": "A_anchor", "seed": 73,
         "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest", "text_normalization": "minimal_twitter",
         "base_id": "base_twitter_roberta_base_minimal"},
        {"model_id": "ft_11_b_dynabench_s17", "kind": "fine_tuned", "source": "pool", "component": "B_dynabench", "seed": 17,
         "checkpoint": "facebook/roberta-hate-speech-dynabench-r4-target", "text_normalization": "minimal_twitter",
         "base_id": "base_dynabench"},
        {"model_id": "ft_11_b_dynabench_s30", "kind": "fine_tuned", "source": "pool", "component": "B_dynabench", "seed": 30,
         "checkpoint": "facebook/roberta-hate-speech-dynabench-r4-target", "text_normalization": "minimal_twitter",
         "base_id": "base_dynabench"},
        {"model_id": "ft_11_b_dynabench_s73", "kind": "fine_tuned", "source": "pool", "component": "B_dynabench", "seed": 73,
         "checkpoint": "facebook/roberta-hate-speech-dynabench-r4-target", "text_normalization": "minimal_twitter",
         "base_id": "base_dynabench"},
        {"model_id": "ft_11_c_hatebert_s17", "kind": "fine_tuned", "source": "pool", "component": "C_hatebert", "seed": 17,
         "checkpoint": "GroNLP/hateBERT", "text_normalization": "minimal_twitter",
         "base_id": "base_hatebert"},
        {"model_id": "ft_11_c_hatebert_s30", "kind": "fine_tuned", "source": "pool", "component": "C_hatebert", "seed": 30,
         "checkpoint": "GroNLP/hateBERT", "text_normalization": "minimal_twitter",
         "base_id": "base_hatebert"},
        {"model_id": "ft_11_c_hatebert_s73", "kind": "fine_tuned", "source": "pool", "component": "C_hatebert", "seed": 73,
         "checkpoint": "GroNLP/hateBERT", "text_normalization": "minimal_twitter",
         "base_id": "base_hatebert"},
        {"model_id": "ft_11_e_twitter_large_s17", "kind": "fine_tuned", "source": "pool", "component": "E_twitter_large", "seed": 17,
         "checkpoint": "cardiffnlp/twitter-roberta-large-hate-latest", "text_normalization": "minimal_twitter",
         "base_id": "base_twitter_roberta_large"},
        {"model_id": "ft_11_e_twitter_large_s30", "kind": "fine_tuned", "source": "pool", "component": "E_twitter_large", "seed": 30,
         "checkpoint": "cardiffnlp/twitter-roberta-large-hate-latest", "text_normalization": "minimal_twitter",
         "base_id": "base_twitter_roberta_large"},
        {"model_id": "ft_11_e_twitter_large_s73", "kind": "fine_tuned", "source": "pool", "component": "E_twitter_large", "seed": 73,
         "checkpoint": "cardiffnlp/twitter-roberta-large-hate-latest", "text_normalization": "minimal_twitter",
         "base_id": "base_twitter_roberta_large"},
        {"model_id": "base_twitter_roberta_base_identity", "kind": "pretrained_base", "paired_with": "ft_07",
         "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest", "text_normalization": "identity"},
        {"model_id": "base_twitter_roberta_base_minimal", "kind": "pretrained_base", "paired_with": "ft_11_a_anchor_s30",
         "checkpoint": "cardiffnlp/twitter-roberta-base-hate-latest", "text_normalization": "minimal_twitter"},
        {"model_id": "base_dynabench", "kind": "pretrained_base", "paired_with": "ft_11_b_dynabench_s30",
         "checkpoint": "facebook/roberta-hate-speech-dynabench-r4-target", "text_normalization": "minimal_twitter"},
        {"model_id": "base_hatebert", "kind": "pretrained_base", "paired_with": "ft_11_c_hatebert_s30",
         "checkpoint": "GroNLP/hateBERT", "text_normalization": "minimal_twitter"},
        {"model_id": "base_twitter_roberta_large", "kind": "pretrained_base", "paired_with": "ft_11_e_twitter_large_s30",
         "checkpoint": "cardiffnlp/twitter-roberta-large-hate-latest", "text_normalization": "minimal_twitter"},
    ],
    # Mean over attended tokens of every hidden-state output; L2 logistic
    # regression on standardised features; C chosen by CV inside train.
    "probe": {
        "pooling": "mean",
        "c_grid": [0.001, 0.01, 0.1, 1.0],
        "cv_folds": 5,
        "cv_seed": 30,
        "max_iter": 5000,
    },
    # Hewitt and Liang (2019) control, adapted to posts: a seeded bit per word type.
    "control_task": {"seed": 30},
    "bootstrap": {"n_resamples": 1000, "seed": 30, "ci_level": 0.95},
    "identity_direction": {
        "enabled": True,
        "layer": "final",
        "n_random_directions": 10,
        "random_seed": 30,
        "screening_threshold": 0.5,
    },
    "inference": {"batch_size": 64, "use_amp_if_cuda": True},
    "reproduction_tolerance": 0.01,
    "external": {"save_hidden_states": False},
    "error_analysis": {"include_text_preview": False},
}
pk.validate_probe_config(CONFIG)
ctx = nk.bootstrap(CONFIG)


## 3. Load foundation and assert the frozen split

Load the processed dataset and the split assignment written by notebook 00. The kit cross-checks them against each other and the manifest, compares the assignment with the committed mirror, and stops unless the class counts are exactly train 1309/790, validation 281/169, test 280/170 (non-extremist/extremist).


In [ ]:
frame, foundation = nk.load_foundation(ctx)
train_df, val_df, test_df = nk.split_frames(frame)

print("Rows per split:", {"train": len(train_df), "validation": len(val_df), "test": len(test_df)})
print("Input file hashes:", json.dumps(foundation["file_sha256_16"], indent=2))


## 4. Probe targets

Three binary targets per post, for the train and validation splits here; the test targets are computed in section 8, after the test split is opened.

* `extremist_label`: the dataset label.
* `identity_term_present`: at least one match of `data/lexicons/identity_terms.txt`, with RQ4's matching rule (the probe kit's copy is pinned to `tools/lexicons.py` and `tools/identity_fpr.py` on every dataset post).
* `control_word_type`: each lower-cased word type gets a seeded pseudo-random bit; a post's score is the mean bit over its words, and the label is `score > c`, with `c` fitted on train so the control has the label's train base rate. It is learnable only by memorising word types, which is what selectivity subtracts.

Only counts are shown.


In [ ]:
identity_terms, lexicon_record = pk.load_identity_terms(ctx)
control_seed = CONFIG["control_task"]["seed"]
control_cutoff, control_record = pk.control_word_type_cutoff(train_df, control_seed)


def targets_for(split_frame):
    """The three probe targets of one split, as integer arrays."""
    return {
        "extremist_label": split_frame["label"].to_numpy().astype(int),
        "identity_term_present": pk.identity_term_present(ctx, split_frame, identity_terms),
        "control_word_type": pk.control_word_type_labels(ctx, split_frame, control_seed, control_cutoff),
    }


targets = {"train": targets_for(train_df), "validation": targets_for(val_df)}
target_counts = pd.DataFrame(
    {
        split_name: {task: int(values.sum()) for task, values in split_targets.items()}
        for split_name, split_targets in targets.items()
    }
)
print("Control cutoff and base rates (train):", json.dumps(control_record, indent=2))
print("Positive posts per target:")
display(target_counts)


## 5. Models and representations

Definitions, and the checks that tie each entry of `CONFIG["models"]` to committed records:

* The text each model receives is the text its notebook served: 07 the raw post (`identity`), 11's runs notebook 11's `minimal_twitter` normalisation, reproduced here verbatim (its regular expressions are pinned to `tools/model_loaders.py` by `tools/tests/test_probe_kit.py`). A base is served with the inputs of the run it is paired with.
* The checkpoint, normalisation, maximum length, components and pooling weights are read from the committed `best_config.json` of 07 and 11, and the notebook stops if `CONFIG` disagrees with them. The locked thresholds come from the same files.
* Weights are found among the inputs by folder name. 11's runs are loaded in the precision recorded in `run_summary.csv`, as notebook 11 served them; 07 in float32 under float16 autocast, as notebook 07 served it.
* `extract_states` returns, for every hidden-state output (layer 0 is the embedding output), the mean over the attended tokens of each post, plus P(EXTREMIST) for a fine-tuned run.
* `ablated_probabilities` re-scores posts with a hook on the last encoder layer that removes one direction from every token's state; the head is unchanged.

Nothing here touches the test split.


In [ ]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

URL_RE = re.compile(r"(?i)\b(?:https?://|www\.)\S+|\bhttp\S*")
USER_RE = re.compile(r"(?<!\w)@[A-Za-z0-9_]+")
WHITESPACE_RE = re.compile(r"\s+")


def normalize_model_text(value, mode):
    """The string a tokenizer receives for one post (notebook 11's normalisation)."""
    value = str(value)
    if mode == "identity":
        return value
    if mode != "minimal_twitter":
        raise ValueError(f"Unsupported text_normalization_mode: {mode}")
    value = html.unescape(value)
    value = unicodedata.normalize("NFKC", value)
    value = USER_RE.sub("@user", value)
    value = URL_RE.sub("http", value)
    return WHITESPACE_RE.sub(" ", value).strip()


def autocast_context(use_amp):
    """float16 autocast when `use_amp`, otherwise a context that does nothing."""
    if not use_amp:
        return nullcontext()
    return torch.autocast(device_type=DEVICE.type, dtype=torch.float16)


# --- committed records -------------------------------------------------------
SINGLE = CONFIG["sources"]["single"]
POOL = CONFIG["sources"]["pool"]
single_config = pk.committed_best_config(ctx, SINGLE)
pool_config = pk.committed_best_config(ctx, POOL)
pool_hyperparameters = pool_config["hyperparameters"]
locked_thresholds = {
    SINGLE: float(single_config["selected_threshold"]),
    POOL: float(pool_config["selected_threshold"]),
}
pool_components = list(pool_hyperparameters["components"])
pool_weights = np.asarray(pool_hyperparameters["component_weights"], dtype=float)
pool_weights = pool_weights / pool_weights.sum()
logit_clip_eps = float(pool_hyperparameters["pooling"]["logit_clip_eps"])
sigmoid_clip = float(pool_hyperparameters["pooling"]["sigmoid_clip"])

model_specs = {entry["model_id"]: dict(entry) for entry in CONFIG["models"]}
configured_components = sorted(
    {s["component"] for s in model_specs.values() if s["kind"] == "fine_tuned" and s["source"] == "pool"}
)
if configured_components != sorted(pool_components):
    raise ValueError(
        f"CONFIG lists pool components {configured_components}; the committed pool is {pool_components}"
    )

for model_id, spec in model_specs.items():
    if spec["kind"] == "fine_tuned":
        spec["source_technique"] = CONFIG["sources"][spec["source"]]
        if spec["source"] == "single":
            committed = single_config["hyperparameters"]
            committed_checkpoint = committed["checkpoint"]
            committed_mode = "identity"
            spec["max_length"] = int(committed["max_length"])
        else:
            committed = pool_hyperparameters["component_run_configs"][spec["component"]]
            committed_checkpoint = committed["checkpoint"]
            committed_mode = pool_hyperparameters["text_normalization_mode"]
            spec["max_length"] = int(committed["max_length"])
        if (spec["checkpoint"], spec["text_normalization"]) != (committed_checkpoint, committed_mode):
            raise ValueError(
                f"{model_id}: CONFIG says {spec['checkpoint']} / {spec['text_normalization']}, the "
                f"committed record says {committed_checkpoint} / {committed_mode}"
            )
for model_id, spec in model_specs.items():
    if spec["kind"] == "pretrained_base":
        paired = model_specs[spec["paired_with"]]
        if (paired["checkpoint"], paired["text_normalization"]) != (spec["checkpoint"], spec["text_normalization"]):
            raise ValueError(f"{model_id}: its checkpoint or inputs differ from {spec['paired_with']}")
        spec["max_length"] = paired["max_length"]
        spec["source_technique"] = ""
        spec["component"] = ""
        spec["seed"] = float("nan")
        spec["base_id"] = ""
    elif model_specs.get(spec["base_id"], {}).get("kind") != "pretrained_base":
        raise ValueError(f"{model_id}: base_id {spec['base_id']} is not a pretrained base in CONFIG")


# --- weights among the inputs ------------------------------------------------
def precision_records():
    """{(component, seed): inference_dtype} from 11's run_summary.csv, or {} if not attached."""
    try:
        path = nk.find_input(ctx, "run_summary.csv", prefer="training_logs")
    except nk.InputNotFoundError:
        return {}
    summary = pd.read_csv(path)
    return {(str(r.component), int(r.seed)): str(r.inference_dtype) for r in summary.itertuples()}


run_precisions = precision_records()
for model_id, spec in model_specs.items():
    if spec["kind"] != "fine_tuned":
        continue
    if spec["source"] == "single":
        weights_dir = pk.find_input_dir(ctx, "weights/transformer_model", prefer=SINGLE)
        spec["tokenizer_dir"] = str(weights_dir.parent.parent / "tokenizer")
        spec["inference_dtype"] = "float32"
        spec["precision_source"] = "notebook 07 serves float32 weights under float16 autocast"
        spec["use_amp"] = bool(CONFIG["inference"]["use_amp_if_cuda"] and DEVICE.type == "cuda")
    else:
        weights_dir = pk.find_input_dir(ctx, f"weights/member-{spec['component']}-seed{spec['seed']}", prefer=POOL)
        spec["tokenizer_dir"] = str(weights_dir.parent.parent / "tokenizer" / spec["component"])
        recorded = run_precisions.get((spec["component"], int(spec["seed"])))
        spec["inference_dtype"] = recorded or "float16"
        spec["precision_source"] = "run_summary.csv" if recorded else "not attached; float16 assumed"
        spec["use_amp"] = bool(DEVICE.type == "cuda" and spec["inference_dtype"] == "float16")
    spec["weights_dir"] = str(weights_dir)
    if not Path(spec["tokenizer_dir"]).is_dir():
        raise FileNotFoundError(f"{model_id}: no tokenizer folder next to its weights")
for spec in model_specs.values():
    if spec["kind"] == "pretrained_base":
        spec["inference_dtype"] = "float32"
        spec["precision_source"] = "Hub checkpoint, float32 weights under float16 autocast"
        spec["use_amp"] = bool(CONFIG["inference"]["use_amp_if_cuda"] and DEVICE.type == "cuda")


def load_model(spec):
    """(tokenizer, model, hub_commit) for one entry, on the device, in eval mode."""
    if spec["kind"] == "fine_tuned":
        tokenizer_ = AutoTokenizer.from_pretrained(spec["tokenizer_dir"], use_fast=True)
        model_ = AutoModelForSequenceClassification.from_pretrained(spec["weights_dir"])
        model_ = model_.half() if spec["inference_dtype"] == "float16" else model_.float()
    else:
        tokenizer_ = AutoTokenizer.from_pretrained(spec["checkpoint"], use_fast=True)
        model_ = AutoModel.from_pretrained(spec["checkpoint"]).float()
    model_.to(DEVICE)
    model_.eval()
    return tokenizer_, model_, getattr(model_.config, "_commit_hash", None)


def release(model_):
    """Free one model before the next is loaded."""
    del model_
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def encode(tokenizer_, strings, spec):
    encoded = tokenizer_(
        strings, truncation=True, padding=True, max_length=int(spec["max_length"]), return_tensors="pt"
    )
    return {key: value.to(DEVICE) for key, value in encoded.items()}


def extract_states(tokenizer_, model_, spec, split_frame):
    """(states [n_layers, n_posts, hidden] float32, P(EXTREMIST) or None) for one split."""
    strings = [normalize_model_text(v, spec["text_normalization"]) for v in split_frame["text"].tolist()]
    batch_size = int(CONFIG["inference"]["batch_size"])
    state_chunks, probability_chunks = [], []
    with torch.inference_mode():
        for start in range(0, len(strings), batch_size):
            encoded = encode(tokenizer_, strings[start : start + batch_size], spec)
            with autocast_context(spec["use_amp"]):
                output = model_(**encoded, output_hidden_states=True)
            mask = encoded["attention_mask"].unsqueeze(-1).float()
            pooled = torch.stack(
                [(hidden.float() * mask).sum(dim=1) / mask.sum(dim=1) for hidden in output.hidden_states]
            )
            state_chunks.append(pooled.cpu().numpy())
            if spec["kind"] == "fine_tuned":
                probability_chunks.append(torch.softmax(output.logits.float(), dim=-1)[:, 1].cpu().numpy())
    states = np.concatenate(state_chunks, axis=1).astype(np.float32)
    probabilities = np.concatenate(probability_chunks) if probability_chunks else None
    return states, probabilities


def projection_hook(direction):
    """A forward hook that removes `direction` (unit vector) from every token's state."""

    def hook(module, inputs, output):
        hidden = output[0] if isinstance(output, tuple) else output
        as_float = hidden.float()
        projected = as_float - (as_float @ direction).unsqueeze(-1) * direction
        projected = projected.to(hidden.dtype)
        return (projected,) + tuple(output[1:]) if isinstance(output, tuple) else projected

    return hook


def ablated_probabilities(tokenizer_, model_, spec, split_frame, direction):
    """P(EXTREMIST) with `direction` removed at the last encoder layer (None: no hook)."""
    strings = [normalize_model_text(v, spec["text_normalization"]) for v in split_frame["text"].tolist()]
    batch_size = int(CONFIG["inference"]["batch_size"])
    handle = None
    if direction is not None:
        unit = torch.tensor(direction, dtype=torch.float32, device=DEVICE)
        handle = model_.base_model.encoder.layer[-1].register_forward_hook(projection_hook(unit))
    outputs = []
    try:
        with torch.inference_mode():
            for start in range(0, len(strings), batch_size):
                encoded = encode(tokenizer_, strings[start : start + batch_size], spec)
                with autocast_context(spec["use_amp"]):
                    logits = model_(**encoded).logits
                outputs.append(torch.softmax(logits.float(), dim=-1)[:, 1].cpu().numpy())
    finally:
        if handle is not None:
            handle.remove()
    return np.concatenate(outputs) if outputs else np.zeros(0)


def to_log_odds(probability):
    p = np.clip(np.asarray(probability, dtype=float), logit_clip_eps, 1.0 - logit_clip_eps)
    return np.log(p / (1.0 - p))


def pool_runs(probabilities_by_model):
    """Notebook 11's pool: seed-mean log-odds per component, weighted mean, sigmoid."""
    component_log_odds = []
    for component in pool_components:
        runs = [
            to_log_odds(probabilities_by_model[model_id])
            for model_id, spec in model_specs.items()
            if spec["kind"] == "fine_tuned" and spec["source"] == "pool" and spec["component"] == component
        ]
        component_log_odds.append(np.mean(runs, axis=0))
    pooled = (pool_weights[:, None] * np.stack(component_log_odds)).sum(axis=0)
    return 1.0 / (1.0 + np.exp(-np.clip(pooled, -sigmoid_clip, sigmoid_clip)))


TASKS = list(pk.PROBE_TASKS)
print("Device:", DEVICE)
print("Locked thresholds:", locked_thresholds)
print("Pool components and weights:", dict(zip(pool_components, pool_weights.round(4).tolist())))
display(
    pd.DataFrame(
        [
            {key: spec.get(key) for key in ("model_id", "kind", "component", "seed", "checkpoint",
                                            "text_normalization", "max_length", "inference_dtype", "base_id")}
            for spec in model_specs.values()
        ]
    )
)


## 6. Representations on train and validation

Load each model once, extract the mean-pooled states of every layer for the train and validation posts, and release it. Every fine-tuned model must reproduce its committed validation probabilities: 07 directly, 11 by pooling its 12 runs as notebook 11 does. The notebook stops otherwise, before any probe is fitted.


In [ ]:
representations = {}
validation_probabilities = {}
model_records = {}
verification = []
extraction_start = time.time()
for model_id, spec in model_specs.items():
    started = time.time()
    tokenizer_, model_, hub_commit = load_model(spec)
    train_states, _ = extract_states(tokenizer_, model_, spec, train_df)
    val_states, val_probability = extract_states(tokenizer_, model_, spec, val_df)
    representations[model_id] = {"train": train_states, "validation": val_states}
    if val_probability is not None:
        validation_probabilities[model_id] = val_probability
    model_records[model_id] = {
        "kind": spec["kind"],
        "checkpoint": spec["checkpoint"],
        "hub_commit": hub_commit,
        "weights_dir": spec.get("weights_dir"),
        "inference_dtype": spec["inference_dtype"],
        "precision_source": spec["precision_source"],
        "text_normalization": spec["text_normalization"],
        "max_length": spec["max_length"],
        "n_layers": int(train_states.shape[0]),
        "hidden_size": int(train_states.shape[2]),
    }
    if CONFIG["external"]["save_hidden_states"] and spec["kind"] == "fine_tuned":
        for split_name, split_frame, split_states in (("train", train_df, train_states), ("validation", val_df, val_states)):
            np.savez_compressed(
                ctx.external("hidden_states") / f"{model_id}__{split_name}.npz",
                states=split_states.astype(np.float16),
                order=split_frame["row_id"].to_numpy().astype(str),
            )
    release(model_)
    print(
        f"{model_id}: {train_states.shape[0]} layers x {train_states.shape[2]} dims, "
        f"{time.time() - started:.0f} s"
    )

verification.append(pk.verify_reproduction(ctx, SINGLE, val_df, validation_probabilities["ft_07"],
                                           tolerance=CONFIG["reproduction_tolerance"]))
verification.append(pk.verify_reproduction(ctx, POOL, val_df, pool_runs(validation_probabilities),
                                           tolerance=CONFIG["reproduction_tolerance"]))
print(f"Extraction time: {(time.time() - extraction_start) / 60:.1f} minutes")


## 7. Probe fit on train

For every model run, layer and task: standardise on the train split, then fit an L2 logistic-regression probe whose `C` is chosen from the grid by 5-fold stratified cross-validated ROC-AUC inside the train split (ties to the smaller `C`), refitted on the whole train split. The scaler is fitted once per layer on the whole train split. Validation is then scored with bootstrap intervals and reported; it selects nothing. The train and validation states are freed at the end of this section.


In [ ]:
probe_settings = CONFIG["probe"]
c_grid = sorted(float(c) for c in probe_settings["c_grid"])
probes = {}
validation_scores = {}
probe_rows = []


def fit_probe(features, labels):
    """C by cross-validated ROC-AUC inside train, then a refit on all of train."""
    search = LogisticRegressionCV(
        Cs=c_grid,
        cv=StratifiedKFold(n_splits=int(probe_settings["cv_folds"]), shuffle=True, random_state=int(probe_settings["cv_seed"])),
        scoring="roc_auc",
        penalty="l2",
        solver="lbfgs",
        max_iter=int(probe_settings["max_iter"]),
        n_jobs=-1,
        refit=True,
    )
    search.fit(features, labels)
    mean_scores = search.scores_[1].mean(axis=0)
    best = int(np.argmax(mean_scores))
    return {
        "c": float(search.Cs_[best]),
        "cv_auc": float(mean_scores[best]),
        "coef": search.coef_[0].astype(np.float64),
        "intercept": float(search.intercept_[0]),
    }


def probe_probability(probe, scaler_record, features):
    """The probe's P(target) for raw features of one layer."""
    standardised = (features - scaler_record["mean"]) / scaler_record["scale"]
    return 1.0 / (1.0 + np.exp(-(standardised @ probe["coef"] + probe["intercept"])))


def probe_row(model_id, layer, task, metrics, probe):
    """One row of rq3_layer_probes.csv."""
    spec = model_specs[model_id]
    record = model_records[model_id]
    row = {
        "model_id": model_id,
        "model_kind": spec["kind"],
        "source_technique": spec["source_technique"],
        "component": spec["component"],
        "checkpoint": spec["checkpoint"],
        "seed": spec["seed"],
        "text_normalization": spec["text_normalization"],
        "n_layers": record["n_layers"],
        "hidden_size": record["hidden_size"],
        "layer": layer,
        "task": task,
        "probe_c": probe["c"],
        "cv_auc_train": probe["cv_auc"],
    }
    row.update(metrics)
    return row


scalers = {}
fit_start = time.time()
for model_id in model_specs:
    started = time.time()
    states = representations[model_id]
    for layer in range(states["train"].shape[0]):
        scaler = StandardScaler().fit(states["train"][layer])
        scaler_record = {"mean": scaler.mean_, "scale": scaler.scale_}
        scalers[(model_id, layer)] = scaler_record
        train_features = scaler.transform(states["train"][layer])
        for task in TASKS:
            probe = fit_probe(train_features, targets["train"][task])
            probes[(model_id, layer, task)] = probe
            scores = probe_probability(probe, scaler_record, states["validation"][layer])
            validation_scores[(model_id, layer, task)] = scores
            metrics = pk.score_probe(ctx, val_df, targets["validation"][task], scores)
            probe_rows.append(probe_row(model_id, layer, task, metrics, probe))
    print(f"{model_id}: probes fitted in {time.time() - started:.0f} s")

del representations
gc.collect()
print(f"Probe fit time: {(time.time() - fit_start) / 60:.1f} minutes")
validation_table = pd.DataFrame(probe_rows)
print("Validation ROC-AUC by layer: extremist_label")
display(
    validation_table[validation_table["task"] == "extremist_label"]
    .pivot_table(index="model_id", columns="layer", values="roc_auc")
    .round(3)
)


## 8. Single test evaluation

Open the test split, once. Then, one model at a time: extract the test states, check that each fine-tuned model reproduces its committed test probabilities, and score every probe on the test posts with bootstrap intervals (1000 resamples, the same draws for every model, layer and task). For each fine-tuned run, remove its final-layer identity-probe direction, and 10 random unit directions, from the last encoder layer and re-score the test posts that contain an identity term. A hook with the zero vector must leave the probabilities unchanged, which checks the hook itself. Finally the contrasts: selectivity (task minus control) and fine-tuned minus base, with paired intervals, on validation and test.


In [ ]:
pk.begin_test_evaluation(ctx)
targets["test"] = targets_for(test_df)
test_scores = {}
test_probabilities = {}
identity_mask = targets["test"]["identity_term_present"] == 1
identity_posts = test_df[identity_mask].reset_index(drop=True)
direction_settings = CONFIG["identity_direction"]
random_generator = np.random.default_rng(int(direction_settings["random_seed"]))
random_directions = {}
ablation_raw = {}

evaluation_start = time.time()
for model_id, spec in model_specs.items():
    started = time.time()
    tokenizer_, model_, _ = load_model(spec)
    test_states, test_probability = extract_states(tokenizer_, model_, spec, test_df)
    n_layers = test_states.shape[0]
    for layer in range(n_layers):
        for task in TASKS:
            scores = probe_probability(probes[(model_id, layer, task)], scalers[(model_id, layer)], test_states[layer])
            test_scores[(model_id, layer, task)] = scores
            metrics = pk.score_probe(ctx, test_df, targets["test"][task], scores)
            probe_rows.append(probe_row(model_id, layer, task, metrics, probes[(model_id, layer, task)]))
    if CONFIG["external"]["save_hidden_states"] and spec["kind"] == "fine_tuned":
        np.savez_compressed(
            ctx.external("hidden_states") / f"{model_id}__test.npz",
            states=test_states.astype(np.float16),
            order=test_df["row_id"].to_numpy().astype(str),
        )
    if spec["kind"] == "fine_tuned":
        test_probabilities[model_id] = test_probability
        if direction_settings["enabled"]:
            hidden_size = test_states.shape[2]
            if hidden_size not in random_directions:
                draws = random_generator.normal(size=(int(direction_settings["n_random_directions"]), hidden_size))
                random_directions[hidden_size] = draws / np.linalg.norm(draws, axis=1, keepdims=True)
            final = n_layers - 1
            raw_direction = probes[(model_id, final, "identity_term_present")]["coef"] / scalers[(model_id, final)]["scale"]
            identity_direction = raw_direction / np.linalg.norm(raw_direction)
            before = ablated_probabilities(tokenizer_, model_, spec, identity_posts, None)
            unchanged = ablated_probabilities(tokenizer_, model_, spec, identity_posts, np.zeros(hidden_size))
            if np.abs(unchanged - before).max() > 1e-6:
                raise RuntimeError(f"{model_id}: the projection hook changes outputs with a zero direction")
            ablation_raw[model_id] = {
                "before": before,
                "identity_probe": ablated_probabilities(tokenizer_, model_, spec, identity_posts, identity_direction),
                "random": np.stack(
                    [ablated_probabilities(tokenizer_, model_, spec, identity_posts, d) for d in random_directions[hidden_size]]
                ),
            }
    release(model_)
    print(f"{model_id}: test scored in {time.time() - started:.0f} s")

verification.append(pk.verify_reproduction(ctx, SINGLE, test_df, test_probabilities["ft_07"],
                                           tolerance=CONFIG["reproduction_tolerance"]))
verification.append(pk.verify_reproduction(ctx, POOL, test_df, pool_runs(test_probabilities),
                                           tolerance=CONFIG["reproduction_tolerance"]))

# --- contrasts ---------------------------------------------------------------
contrast_rows = []
split_inputs = {
    "validation": (val_df, validation_scores),
    "test": (test_df, test_scores),
}
for split_name, (split_frame, split_scores) in split_inputs.items():
    split_targets = targets[split_name]
    for model_id, spec in model_specs.items():
        for layer in range(model_records[model_id]["n_layers"]):
            for task in ("extremist_label", "identity_term_present"):
                difference = pk.paired_auc_difference(
                    ctx, split_frame,
                    split_targets[task], split_scores[(model_id, layer, task)],
                    split_targets["control_word_type"], split_scores[(model_id, layer, "control_word_type")],
                )
                difference.pop("split")
                contrast_rows.append({"contrast": "selectivity", "model_id": model_id, "reference_id": model_id,
                                      "layer": layer, "task": task, "split": split_name, **difference})
            if spec["kind"] != "fine_tuned":
                continue
            for task in TASKS:
                difference = pk.paired_auc_difference(
                    ctx, split_frame,
                    split_targets[task], split_scores[(model_id, layer, task)],
                    split_targets[task], split_scores[(spec["base_id"], layer, task)],
                )
                difference.pop("split")
                contrast_rows.append({"contrast": "fine_tuned_minus_base", "model_id": model_id,
                                      "reference_id": spec["base_id"], "layer": layer, "task": task,
                                      "split": split_name, **difference})

# --- identity-direction ablation ----------------------------------------------
ablation_rows = []
populations = {"non_extremist": 0, "extremist": 1}
identity_labels = identity_posts["label"].to_numpy()


def add_ablation_rows(model_id, source_technique, seed, before, identity_after, random_after, threshold, kind):
    """Two rows (identity_probe, random) per population for one model or the pool."""
    for population, label in populations.items():
        keep = identity_labels == label
        population_frame = identity_posts[keep].reset_index(drop=True)
        random_deltas = [float((before[keep] - draw[keep]).mean()) for draw in random_after]
        random_p95 = float(np.percentile(random_deltas, 95))
        for direction, after in (("identity_probe", identity_after), ("random", random_after.mean(axis=0))):
            ablation_rows.append(
                pk.direction_ablation_row(
                    ctx, population_frame, before[keep], after[keep], threshold, kind,
                    {"model_id": model_id, "source_technique": source_technique, "seed": seed,
                     "population": population, "direction": direction},
                    random_delta_p95=random_p95,
                    n_random_directions=len(random_after),
                )
            )


if direction_settings["enabled"]:
    for model_id, raw in ablation_raw.items():
        spec = model_specs[model_id]
        if spec["source"] == "single":
            threshold, kind = locked_thresholds[SINGLE], "locked"
        else:
            threshold, kind = float(direction_settings["screening_threshold"]), "screening"
        add_ablation_rows(model_id, spec["source_technique"], spec["seed"], raw["before"],
                          raw["identity_probe"], raw["random"], threshold, kind)
    pooled_runs = [m for m, s in model_specs.items() if s["kind"] == "fine_tuned" and s["source"] == "pool"]
    pooled_before = pool_runs({m: ablation_raw[m]["before"] for m in pooled_runs})
    pooled_identity = pool_runs({m: ablation_raw[m]["identity_probe"] for m in pooled_runs})
    pooled_random = np.stack(
        [pool_runs({m: ablation_raw[m]["random"][k] for m in pooled_runs})
         for k in range(int(direction_settings["n_random_directions"]))]
    )
    add_ablation_rows("11_pooled", POOL, float("nan"), pooled_before, pooled_identity, pooled_random,
                      locked_thresholds[POOL], "locked")

print(f"Test evaluation time: {(time.time() - evaluation_start) / 60:.1f} minutes")
print("Identity-term test posts by label:", {p: int((identity_labels == l).sum()) for p, l in populations.items()})
print("Reproduction checks:")
display(pd.DataFrame(verification))


## 9. Probe tables

Holm-adjust the preregistered family (the `identity_probe` rows of 07 and of 11 pooled, at their locked thresholds: four rows); every other ablation row is descriptive and has no Holm value. Then write the three tables and the run record with `pk.export_probe_tables`, which validates each against its contract first and refuses a per-post column or a row id anywhere. The tables shown are per-layer metrics.


In [ ]:
probe_table = pd.DataFrame(probe_rows)[list(pk.PROBE_COLUMNS)]
contrast_table = pd.DataFrame(contrast_rows)[list(pk.CONTRAST_COLUMNS)]
ablation_table = None
if direction_settings["enabled"]:
    ablation_table = pd.DataFrame(ablation_rows)[list(pk.ABLATION_COLUMNS)]
    family = (ablation_table["direction"] == "identity_probe") & (ablation_table["threshold_kind"] == "locked")
    for raw_column, holm_column in (("wilcoxon_p", "wilcoxon_p_holm"), ("mcnemar_p", "mcnemar_p_holm")):
        values = ablation_table.loc[family, raw_column].to_numpy(dtype=float)
        adjusted = np.full(len(values), np.nan)
        finite = np.isfinite(values)
        adjusted[finite] = pk.holm_adjust(values[finite])
        ablation_table.loc[family, holm_column] = adjusted

split_counts = {
    split_name: {task: int(values.sum()) for task, values in split_targets.items()}
    for split_name, split_targets in targets.items()
}
run_record = {
    "preregistration": "docs/preregistrations/WP9_layer_probes.md",
    "models": model_records,
    "locked_thresholds": locked_thresholds,
    "pool": {"components": pool_components, "weights": pool_weights.tolist(),
             "logit_clip_eps": logit_clip_eps, "sigmoid_clip": sigmoid_clip},
    "verification": verification,
    "lexicon": lexicon_record,
    "control_task": control_record,
    "target_positive_counts": split_counts,
    "identity_test_posts": {p: int((identity_labels == l).sum()) for p, l in populations.items()},
    "holm_family": "identity_probe rows at a locked threshold (07 and 11 pooled)",
    "environment": nk.describe_environment(),
}
table_paths = pk.export_probe_tables(ctx, probe_table, contrast_table, ablation_table, run_record)

test_probes = probe_table[probe_table["split"] == "test"]
for task in TASKS:
    print(f"Test ROC-AUC by layer: {task}")
    display(test_probes[test_probes["task"] == task].pivot_table(index="model_id", columns="layer", values="roc_auc").round(3))
if ablation_table is not None:
    display(ablation_table[family])


## 10. Interpretation

This section is written before the run. Numbers are not copied here; they live in the tables, and `tools/layer_probes_report.py --summary` prints the per-model summary and the preregistered reading of the ablation.

**Reading the outputs.** A probe's test AUC says how well the target can be read off one layer's mean-pooled representation by a linear model fitted on train. Selectivity subtracts the word-type control on the same layer, so a high AUC that the control matches says the layer keeps word identity, not that it encodes the target. The fine-tuning contrast compares a run with the checkpoint it started from, on the same posts and inputs: an interval above zero at a layer means fine-tuning on this dataset made the label (or identity presence) more linearly decodable there. The rules for each statement are fixed in the preregistration: no family correction on probe metrics, statements by layer range, seed 30 for 11's components with the agreement of the other seeds.

**The ablation.** Removing the identity direction asks whether the information a linear probe uses to detect an identity term also moves p(extremist). For 07 and 11 pooled, a Holm-adjusted Wilcoxon p below 0.05 together with a mean shift above the 95th percentile of 10 random directions is read as "the identity direction carries label-relevant signal"; otherwise "no evidence" or "partial". A false-positive change is always reported with the extremist-post change, and none of this is read as the model treating identity terms as evidence of extremism (decision 5, WP7 and WP7b).

**Limitations.**
- A linear probe measures what is decodable, not what the model uses. Only the ablation intervenes, on one linear direction at one layer; a network can carry the same information non-linearly or elsewhere.
- Mean pooling over tokens is one choice of post representation; the first-token state is not probed.
- The base checkpoints are downloaded at run time; the recorded Hub commits show whether they differ from what notebooks 07 and 11 started from.
- Models are served in float16, as their notebooks served them; the reproduction checks bound the difference.
- The word-type control adapts a token-level method to posts.
- One dataset and a 450-post test split: the intervals are wide, and a difference whose interval includes zero is not a difference.
- Labels are post-level and features are text only. This describes representations of posts, not accounts, intent or radicalisation over time.


## 11. Package outputs

Inventory the external files with their hashes, check that nothing text-bearing or per-post sits in the committed tree, and zip the files that go into the repository. The zip unpacks at the repository root.

**After the run.**
1. Unzip `13_TRANSFORMER-LAYER-PROBES_repo_files.zip` at the repository root (it holds `results_summary/rq/` files only).
2. Run `python3 tools/layer_probes_report.py --check`, then `--write`, and `python3 tools/scan_text_leakage.py`.
3. Only with `save_hidden_states`: upload `external/13_TRANSFORMER-LAYER-PROBES/hidden_states/` to a Kaggle Dataset and record each file with the `run_manifest.py add-asset` command printed below, against the technique whose weights produced it.
4. Before committing this notebook with its outputs, run `tools/check_notebook.py --notebook notebooks/13_TRANSFORMER-LAYER-PROBES.ipynb`.


In [ ]:
cache_dir = ctx.external_dir / "hidden_states"
cache_files = sorted(cache_dir.glob("*.npz")) if cache_dir.exists() else []
for cache_file in cache_files:
    source_technique = model_specs[cache_file.name.split("__")[0]]["source_technique"]
    print(
        f"python3 tools/run_manifest.py add-asset --technique {source_technique} --kind other "
        f"--location kaggle://datasets/<user>/<slug>/v1/hidden_states/{cache_file.name} "
        f"--sha256 {nk.sha256_file(cache_file)} --size-bytes {cache_file.stat().st_size} "
        f'--contains-text --description "WP9 mean-pooled hidden states (13_TRANSFORMER-LAYER-PROBES)"'
    )
print(f"Hidden-state cache files: {len(cache_files)}")


In [ ]:
nk.finalize(ctx)
